In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

PLL_DIR = Path.cwd().resolve().parents[2] / "results" / "pll"  # notebook lives in sw/tools/notebooks/
sys.path.insert(0, str(PLL_DIR.parents[1]))  # repo root
from sw.lib.pll_util import bin_phase_noise

RESULT_CSV = None  # e.g. PLL_DIR / "pll_analyzer_20261019_150000.csv"; None = newest run
LABELS = None  # the two measurements to compare, e.g. ["FSW 1", "FSW 2"]; None = the first two measured ones

if RESULT_CSV is None:
    RESULT_CSV = sorted((p for p in PLL_DIR.glob("pll_analyzer_*.csv") if not p.stem.endswith("_traces")),
                        key=lambda p: p.stat().st_mtime)[-1]
RESULT_CSV = Path(RESULT_CSV)
runs = pd.read_csv(RESULT_CSV)
traces = pd.read_csv(RESULT_CSV.with_name(RESULT_CSV.stem + "_traces.csv"))
if LABELS is None:
    LABELS = list(runs.loc[runs["status"] == "ok", "label"][:2])
A, B = (runs[runs["label"] == label].iloc[0] for label in LABELS)
print(f"{RESULT_CSV.name}: {A.label} ({A.analyzer_name}) vs {B.label} ({B.analyzer_name}); VCO {A.f_vco / 1e6:.1f} MHz, "
      f"pad {A.f_out_meas / 1e6:.4f} MHz (/{A.total_div:.0f}), VCO coarse {A.vco_tune_coarse:.0f} "
      f"min {A.vco_current_min:.0f} max {A.vco_current_max:.0f}, Icp {A.set_current:.0f} C1 {A.set_c1:.0f} "
      f"C2 {A.set_c2:.0f} R1 {A.set_r1:.0f}")

# The two measurements side by side: carrier, phase-noise markers on the pad, RMS jitter; B - A in the last column
COMPARE = {"time": "time", "status": "status", "carrier [MHz]": "f_out_meas", "carrier [dBm]": "carrier_dbm",
           "detector": "detector", "average type": "average_type", "video filter": "video_filter"}
COMPARE.update({f"L {c[6:]} Hz [dBc/Hz]": c for c in runs.columns if c.startswith("L_out_")})
COMPARE.update({"jitter markers [ps]": "time_jitter_ps", "jitter curve [ps]": "time_jitter_trace_ps",
                "jitter noise [ps]": "time_jitter_noise_ps", "jitter spurs [ps]": "time_jitter_spur_ps",
                "jitter markers 1k-10M [ps]": "time_jitter_common_ps",
                "jitter curve 1k-10M [ps]": "time_jitter_trace_common_ps", "spurs": "spurs"})
table = pd.DataFrame({label: [r[c] for c in COMPARE.values()] for label, r in zip(LABELS, (A, B))},
                     index=list(COMPARE))
table.loc["carrier [MHz]"] = table.loc["carrier [MHz]"] / 1e6
table["B - A"] = [b - a if isinstance(a, (int, float)) and isinstance(b, (int, float)) else ""
                  for a, b in zip(table[LABELS[0]], table[LABELS[1]])]
display(table)

pll_analyzer_20261019_141836.csv: FSW 1 (Rohde & Schwarz FSW43 Signal Analyzer) vs FSW 2 (Rohde & Schwarz FSW43 Signal Analyzer); VCO 4096.0 MHz, pad 64.0000 MHz (/64), VCO coarse 0 min 8 max 0, Icp 7 C1 0 C2 0 R1 7


,FSW 1,FSW 2,B - A
time,2026-10-19T14:18:36,2026-10-19T14:19:00,
status,ok,ok,
carrier [MHz],63.999956,63.999956,0.0
carrier [dBm],-8.304464,-8.313547,-0.009083
detector,RMS,RMS,
average type,POW,POW,
video filter,LIN,LIN,
L 1000 Hz [dBc/Hz],-83.836679,-85.503877,-1.667198
L 10000 Hz [dBc/Hz],-93.875813,-94.677351,-0.801539
L 100000 Hz [dBc/Hz],-100.380225,-99.245853,1.134372


In [3]:
INK, GRID = "#52514e", "#e4e3df"
COLORS = ["#2a78d6", "#eb6834"]  # A, B
OFFSET_AXIS = dict(type="log", tickvals=[1e2, 1e3, 1e4, 1e5, 1e6, 1e7, 1e8],
                   ticktext=["100 Hz", "1 kHz", "10 kHz", "100 kHz", "1 MHz", "10 MHz", "100 MHz"],
                   minor=dict(showgrid=True, gridcolor="#f1f0ed"))
N_BINS = 200  # points per plotted phase-noise curve (linear power average, display only)


def trace_of(label, kind):
    return traces[(traces["label"] == label) & (traces["kind"] == kind)].sort_values("freq")


def save(fig, name):
    """Interactive HTML and a static PNG (kaleido) next to the CSV, then show the figure."""
    base = RESULT_CSV.with_name(f"{RESULT_CSV.stem}_{name}")
    fig.write_html(base.with_suffix(".html"))
    try:
        fig.write_image(base.with_suffix(".png"), width=1100, height=fig.layout.height or 600, scale=2)
        print(f"saved {base}.html and .png")
    except Exception as e:  # kaleido >= 1.0 needs a Chrome/Chromium install (`plotly_get_chrome`)
        print(f"saved {base}.html; PNG export failed: {e}")
    fig.show()

In [4]:
F_start, F_stop = 1e3, 10e6  # offset range shown [Hz]

# Phase noise on the pad (full curve) of both measurements on top of each other, their markers as symbols, and below
# the difference B - A per bin (same N_BINS log-spaced bins for both)
edges = np.geomspace(F_start, F_stop, N_BINS + 1)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05, row_heights=[0.7, 0.3])
binned = {}
for label, r, color in zip(LABELS, (A, B), COLORS):
    d = trace_of(label, "L_trace")
    d = d[(d["freq"] >= F_start) & (d["freq"] <= F_stop)]
    freq, level = bin_phase_noise(d["freq"], d["level_dbm"], edges)
    binned[label] = pd.Series(level, index=np.digitize(freq, edges))
    fig.add_scatter(x=freq, y=level, mode="lines", name=f"{label}: full curve", line=dict(color=color, width=1.5),
                    hovertemplate=f"{label}<br>%{{x:.3s}}Hz: %{{y:.1f}} dBc/Hz<extra></extra>", row=1, col=1)
    marks = {float(c[6:]): r[c] for c in runs.columns if c.startswith("L_out_") and pd.notna(r[c])}
    marks = {f: l for f, l in marks.items() if F_start <= f <= F_stop}
    fig.add_scatter(x=list(marks), y=list(marks.values()), mode="markers", name=f"{label}: markers",
                    marker=dict(color=color, size=10, symbol="diamond-open", line=dict(width=2)),
                    hovertemplate=f"{label} marker<br>%{{x:.3s}}Hz: %{{y:.1f}} dBc/Hz<extra></extra>", row=1, col=1)
diff = (binned[LABELS[1]] - binned[LABELS[0]]).dropna()
fig.add_scatter(x=np.sqrt(edges[diff.index - 1] * edges[diff.index]), y=diff.values, mode="lines", showlegend=False,
                line=dict(color=INK, width=1.5),
                hovertemplate="%{x:.3s}Hz: %{y:+.2f} dB<extra></extra>", row=2, col=1)
fig.add_hline(y=0, line=dict(color=GRID, width=1), row=2, col=1)
fig.update_layout(title=f"Phase noise on the pad, {LABELS[0]} vs {LABELS[1]}, carrier {A.f_out_meas / 1e6:.3f} MHz"
                        f"<br><sub>{RESULT_CSV.name}</sub>", template="plotly_white", height=750,
                  legend=dict(orientation="h", y=-0.12), hoverlabel=dict(namelength=-1))
fig.update_xaxes(gridcolor=GRID, **OFFSET_AXIS, range=[np.log10(F_start), np.log10(F_stop)])
fig.update_xaxes(title="offset from the carrier", row=2)
fig.update_yaxes(gridcolor=GRID)
fig.update_yaxes(title="L(f) on the pad [dBc/Hz]", row=1)
fig.update_yaxes(title=f"{LABELS[1]} - {LABELS[0]} [dB]", row=2)
save(fig, "phase_noise")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_analyzer_20261019_141836_phase_noise.html and .png


In [5]:
# Averaged spectrum on the pad around the carrier of both measurements, relative to the carrier (trace maximum), in
# the trace's RBW (not binned)
fig = go.Figure()
for label, r, color in zip(LABELS, (A, B), COLORS):
    d = trace_of(label, "spectrum")
    fig.add_scatter(x=d["freq"] - d["center"], y=d["level_dbm"] - d["level_dbm"].max(), mode="lines",
                    name=f"{label} ({r.analyzer_name}), RBW {d['rbw'].iloc[0]:g} Hz",
                    line=dict(color=color, width=1),
                    hovertemplate=f"{label}<br>%{{x:.4s}}Hz: %{{y:.1f}} dBc<extra></extra>")
fig.update_layout(title=f"Spectrum on the pad around the carrier, {LABELS[0]} vs {LABELS[1]}"
                        f"<br><sub>{RESULT_CSV.name}</sub>", template="plotly_white", height=550,
                  legend=dict(orientation="h", y=-0.15), hoverlabel=dict(namelength=-1))
fig.update_xaxes(title="offset from the carrier", type="linear", ticksuffix="Hz", exponentformat="SI", gridcolor=GRID)
fig.update_yaxes(title="level relative to the carrier [dBc in RBW]", gridcolor=GRID)
save(fig, "spectrum")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_analyzer_20261019_141836_spectrum.html and .png
